<a href="https://colab.research.google.com/github/crisMildenberger/AgendaElectronica-Form-/blob/master/Proyecto_integrador_Churn_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto integrador — Predicción de Churn de clientes

**Objetivo:** construir un pipeline reproducible de clasificación binaria que cubra carga, control de calidad, EDA, ETL/feature engineering, preprocesamiento, baseline, búsqueda de hiperparámetros, evaluación e interpretación.

**Dataset:** Telco Customer Churn, originalmente publicado como muestra de IBM y distribuido públicamente en Kaggle. Cada fila representa un cliente y `Churn` indica si abandonó durante el último mes. El archivo contiene 7.043 registros y 21 columnas.  
Fuente documental: IBM y la ficha pública de Kaggle.

> **Importante:** los resultados numéricos de este notebook se generan al ejecutarlo. No se fijan valores "de ejemplo" para evitar presentar como resultados propios cifras que todavía no fueron calculadas.

## 1. Contexto del problema

Una empresa de telecomunicaciones quiere identificar clientes con mayor riesgo de abandono (`churn`) para priorizar acciones de retención.

### Pregunta de negocio

> ¿Podemos construir un modelo que estime la probabilidad de churn de cada cliente y permita priorizar campañas de retención?

### Hipótesis de trabajo

Variables como antigüedad (`tenure`), tipo de contrato, método de pago, servicio de Internet y cargos mensuales podrían contener información útil para discriminar clientes con distinto riesgo.

### KPI de negocio

- **Churn rate:** proporción de clientes que abandonan.
- **Retención:** proporción de clientes que permanecen.
- **Clientes en riesgo detectados:** verdaderos positivos.
- **Ingreso potencial retenido:** clientes recuperados × ingreso mensual relevante.
- **Costo de campaña:** clientes contactados × costo por contacto.
- **ROI de retención:** beneficio incremental / costo de campaña.

Las cifras monetarias deben definirse con datos reales de la empresa. El dataset es un conjunto de muestra, por lo que no debe interpretarse como facturación real en ARS.

In [ ]:
# 2. Configuración reproducible

import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Python:", sys.version)
print("scikit-learn:", sklearn_version)

RANDOM_STATE = 42

## 3. Carga de datos
La fuente original del conjunto es IBM; la copia utilizada en este ejemplo está publicada en Kaggle

In [ ]:
# Opción reproducible: cargar desde una copia pública del CSV.
# Si trabajás con el archivo descargado en Colab, reemplazá URL por la ruta local.

URL = "https://raw.githubusercontent.com/aiplanethub/Datasets/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(URL)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
display(df.head())

## 4. Control de calidad inicial

Antes de modelar se revisan:

1. dimensiones;
2. tipos de datos;
3. duplicados;
4. valores faltantes;
5. cardinalidad;
6. consistencia de variables;
7. distribución de la variable objetivo.

`TotalCharges` suele llegar como texto en este dataset y contiene valores vacíos para algunos clientes nuevos; por eso debe convertirse a numérico de forma explícita.

In [ ]:
# Inspección estructural
display(df.info())

print("\nDuplicados completos:", df.duplicated().sum())

# Cardinalidad y nulos
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(dropna=False),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

display(quality)

# Conversión necesaria
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing después de convertir TotalCharges:")
display(df.isna().sum().sort_values(ascending=False).head())

### 4.1. Missingness

Los valores vacíos de `TotalCharges` están asociados a clientes con poca antigüedad, por lo que la imputación debe hacerse dentro del pipeline y sin utilizar `Churn` para calcularla.

In [ ]:
plt.figure(figsize=(12, 5))
msno.matrix(df)
plt.title("Patrón de valores faltantes")
plt.show()

missing_summary = (
    df.isna().sum()
      .rename("missing")
      .to_frame()
      .assign(pct=lambda x: x["missing"] / len(df) * 100)
      .query("missing > 0")
      .sort_values("missing", ascending=False)
)

display(missing_summary)

## 5. EDA — Variable objetivo

La primera pregunta es si existe desbalance de clases.

Se calcula el **churn rate** como proporción de `Yes`.

In [ ]:
target = "Churn"

target_counts = df[target].value_counts(dropna=False)
target_pct = df[target].value_counts(normalize=True, dropna=False) * 100

eda_target = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct.round(2)
})
display(eda_target)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target)
plt.title("Distribución de Churn")
plt.xlabel("Churn")
plt.ylabel("Clientes")
plt.show()

## 6. EDA — Variables numéricas

Se observan las distribuciones de:

- `tenure`: antigüedad en meses;
- `MonthlyCharges`: cargo mensual;
- `TotalCharges`: cargos acumulados.

Los boxplots por clase permiten detectar diferencias de distribución y posibles outliers.

In [ ]:
numeric_eda = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, numeric_eda):
    sns.boxplot(data=df, x=target, y=col, ax=ax)
    ax.set_title(f"{col} vs Churn")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, numeric_eda):
    sns.histplot(
        data=df,
        x=col,
        hue=target,
        kde=True,
        stat="density",
        common_norm=False,
        ax=ax
    )
    ax.set_title(f"Distribución de {col}")

plt.tight_layout()
plt.show()

## 7. EDA — Variables categóricas

Para evitar gráficos excesivamente largos se seleccionan variables de negocio relevantes.

El análisis busca diferencias de tasa de churn entre categorías. Estas diferencias son **descriptivas**, no implican causalidad.

In [ ]:
categorical_eda = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "TechSupport",
    "OnlineSecurity",
    "PaperlessBilling",
]

for col in categorical_eda:
    rate = (
        df.groupby(col, dropna=False)[target]
          .apply(lambda s: (s == "Yes").mean() * 100)
          .sort_values(ascending=False)
    )

    plt.figure(figsize=(8, 4))
    sns.barplot(x=rate.values, y=rate.index)
    plt.title(f"Tasa de churn por {col}")
    plt.xlabel("Churn (%)")
    plt.ylabel(col)
    plt.show()

## 8. EDA — Correlaciones numéricas

La matriz de correlación sirve como diagnóstico inicial entre variables numéricas. No reemplaza el análisis de variables categóricas ni demuestra causalidad.

Se codifica temporalmente `Churn` como 0/1 sólo para esta visualización.

In [ ]:
corr_df = df[numeric_eda].copy()
corr_df["Churn_binary"] = (df["Churn"] == "Yes").astype(int)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre variables numéricas")
plt.show()

# 9. ETL y Feature Engineering

Se crean variables que expresan relaciones más útiles para el modelo:

- `tenure_years`: antigüedad en años.
- `avg_monthly_spend`: cargos acumulados / antigüedad, cuando la antigüedad es positiva.
- `total_services`: cantidad de servicios adicionales contratados.
- `is_month_to_month`: indicador de contrato mensual.
- `high_support`: indicador de soporte técnico contratado.


In [ ]:
def feature_engineering(data):
    data = data.copy()

    # Antigüedad en años
    data["tenure_years"] = data["tenure"] / 12

    # Gasto mensual promedio observado
    data["avg_monthly_spend"] = np.where(
        data["tenure"] > 0,
        data["TotalCharges"] / data["tenure"],
        np.nan
    )

    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
    ]

    # Cuenta "Yes" como servicio; los valores "No internet service" /
    # "No phone service" no cuentan como servicio adicional.
    data["total_services"] = (data[service_cols] == "Yes").sum(axis=1)

    data["is_month_to_month"] = (data["Contract"] == "Month-to-month").astype(int)
    data["high_support"] = (data["TechSupport"] == "Yes").astype(int)

    # ID no entra al modelo
    data = data.drop(columns=["customerID"])

    return data

df_model = feature_engineering(df)

display(df_model.head())
print(df_model.shape)

## 10. Separación de X e y

Separamos el target antes del preprocesamiento.

El split se realiza **antes de entrenar cualquier transformador que aprenda parámetros de los datos**. Esto ayuda a mantener el conjunto de test completamente aislado.

In [ ]:
X = df_model.drop(columns=[target])
y = (df_model[target] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nProporción de churn:")
print("Train:", y_train.mean().round(4))
print("Test :", y_test.mean().round(4))

# 11. Preprocesamiento

### Variables numéricas

1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`

La mediana es menos sensible a valores extremos que la media.

### Variables categóricas

1. `SimpleImputer(strategy="most_frequent")`
2. `OneHotEncoder(handle_unknown="ignore")`



In [ ]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "tenure_years",
    "avg_monthly_spend",
    "total_services",
    "is_month_to_month",
    "high_support",
]

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

def obtener_onehot_compat():
    # scikit-learn >= 1.2 usa sparse_output.
    # Versiones anteriores usaban sparse.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

def construir_preprocesador(
    numeric_features,
    categorical_features
):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", obtener_onehot_compat()),
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])

preprocessor = construir_preprocesador(
    numeric_features,
    categorical_features
)

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

# 12. Baseline — Logistic Regression


In [ ]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_pipeline.fit(X_train, y_train)

y_pred_log = logistic_pipeline.predict(X_test)
y_prob_log = logistic_pipeline.predict_proba(X_test)[:, 1]

baseline_metrics = {
    "ROC AUC": roc_auc_score(y_test, y_prob_log),
    "Accuracy": accuracy_score(y_test, y_pred_log),
    "Precision": precision_score(y_test, y_pred_log),
    "Recall": recall_score(y_test, y_pred_log),
    "F1": f1_score(y_test, y_pred_log),
}

baseline_metrics = pd.Series(baseline_metrics).round(4)
display(baseline_metrics)

print(classification_report(
    y_test,
    y_pred_log,
    target_names=["No churn", "Churn"]
))

# 13. Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", __import__("sklearn").ensemble.RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

rf_metrics = {
    "ROC AUC": roc_auc_score(y_test, y_prob_rf),
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1": f1_score(y_test, y_pred_rf),
}

display(pd.Series(rf_metrics).round(4))

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["No churn", "Churn"]
))

# 14. Comparación inicial

La métrica principal será ROC AUC porque el problema se plantea como priorización/ranking de clientes por riesgo. Sin embargo, para una campaña real también hay que mirar recall y precision de la clase `Churn`.

No existe una única métrica correcta independientemente del costo de negocio.

In [ ]:
comparison = pd.DataFrame({
    "LogisticRegression": baseline_metrics,
    "RandomForest": pd.Series(rf_metrics),
}).T

display(comparison.round(4))

# 15. Validación cruzada estratificada

En lugar de confiar en una sola partición, usamos `StratifiedKFold` para conservar aproximadamente la proporción de churn en cada fold.

La búsqueda de hiperparámetros se realiza únicamente sobre `X_train`/`y_train`. El test final permanece sin tocar hasta la evaluación final.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print("\nMejor ROC AUC promedio CV:")
print(round(grid_search.best_score_, 4))

## 16. Diagnóstico de la búsqueda

Es útil revisar la diferencia entre `mean_train_score` y `mean_test_score`.

Un gap grande puede indicar que el modelo está aprendiendo demasiado bien el conjunto de entrenamiento respecto de los folds de validación.

In [ ]:
results_grid = pd.DataFrame(grid_search.cv_results_)

cols = [
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "mean_train_score",
    "mean_test_score",
    "std_test_score",
]

display(
    results_grid[cols]
    .sort_values("mean_test_score", ascending=False)
    .head(10)
    .round(4)
)

# 17. Evaluación final del modelo seleccionado

Ahora sí se utiliza el conjunto de test, que no participó en la búsqueda de hiperparámetros.

Representa el comportamiento del modelo sobre una muestra que quedó fuera del ajuste.

In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

final_metrics = pd.Series({
    "ROC AUC": roc_auc_score(y_test, y_prob),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
})

display(final_metrics.round(4))

print(classification_report(
    y_test,
    y_pred,
    target_names=["No churn", "Churn"],
    digits=4
))

# 18. Matriz de confusión

- **TP:** churners correctamente detectados.
- **FN:** churners que el modelo no detectó.
- **FP:** clientes que el modelo marcó como riesgo pero no abandonaron.
- **TN:** clientes correctamente clasificados como no churn.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No churn", "Churn"]
)

disp.plot(values_format="d")
plt.title("Matriz de confusión — modelo final")
plt.show()

# 19. Curva ROC

La curva ROC muestra la relación entre sensibilidad (TPR) y tasa de falsos positivos (FPR) a diferentes umbrales.

El AUC resume la capacidad de ranking del modelo.

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_prob,
    name=f"Random Forest (AUC={roc_auc_score(y_test, y_prob):.3f})"
)

plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Curva ROC")
plt.show()

# 20. Umbral de decisión y negocio

`predict()` utiliza por defecto un umbral cercano a 0.5. Pero el umbral óptimo depende del costo de los errores.

Si una campaña de retención es barata, puede ser razonable contactar más clientes y aceptar más falsos positivos. Si el contacto es caro, puede ser necesario priorizar precision.

Por eso conviene separar:

1. **modelo:** estima probabilidad;
2. **política de decisión:** define a quién contactar.

In [ ]:
thresholds = np.arange(0.20, 0.81, 0.05)

threshold_table = []

for threshold in thresholds:
    pred_t = (y_prob >= threshold).astype(int)

    threshold_table.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0),
        "f1": f1_score(y_test, pred_t, zero_division=0),
        "contact_rate": pred_t.mean(),
    })

threshold_table = pd.DataFrame(threshold_table)
display(threshold_table.round(3))

# 21. Interpretabilidad — importancia de variables

In [ ]:
# Recuperar nombres de features después del ColumnTransformer
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.Series(importances, index=feature_names)
      .sort_values(ascending=False)
      .head(20)
)

display(feature_importance.to_frame("importance"))

plt.figure(figsize=(10, 7))
sns.barplot(
    x=feature_importance.values,
    y=feature_importance.index
)
plt.title("Top 20 importancias — Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# 22. KPI de negocio

El dataset permite calcular directamente el churn rate observado.

Los demás KPI económicos requieren supuestos externos. Para evitar inventar una facturación real, se parametriza el escenario.

In [ ]:
churn_rate = y.mean()
retention_rate = 1 - churn_rate

kpis = pd.Series({
    "Churn rate": churn_rate,
    "Retention rate": retention_rate,
    "Customers": len(df_model),
})

display(kpis.round(4))

# Escenario hipotético de negocio
def impacto_retencion(
    clientes,
    churn_rate,
    recall_modelo,
    tasa_recuperacion,
    ingreso_mensual,
    costo_contacto
):
    churners = clientes * churn_rate
    churners_detectados = churners * recall_modelo
    recuperados = churners_detectados * tasa_recuperacion

    ingreso_retenido = recuperados * ingreso_mensual
    clientes_contactados = churners_detectados
    costo_campania = clientes_contactados * costo_contacto

    beneficio_neto = ingreso_retenido - costo_campania
    roi = (
        beneficio_neto / costo_campania
        if costo_campania > 0
        else np.nan
    )

    return pd.Series({
        "Clientes en riesgo": churners,
        "Churners detectados": churners_detectados,
        "Clientes recuperados": recuperados,
        "Ingreso retenido": ingreso_retenido,
        "Costo campaña": costo_campania,
        "Beneficio neto": beneficio_neto,
        "ROI": roi,
    })

# Estos valores son SOLO un escenario ilustrativo.
impacto = impacto_retencion(
    clientes=10000,
    churn_rate=0.045,
    recall_modelo=final_metrics["Recall"],
    tasa_recuperacion=0.40,
    ingreso_mensual=15000,
    costo_contacto=1500
)

display(impacto.round(2))

# 23. Control explícito de riesgos

### Data leakage
- El test queda separado antes del ajuste.
- Imputación y escalado se entrenan dentro del pipeline.
- El target no se utiliza para construir features.
- El `TargetEncoder`, cuando se utiliza, debe estar dentro del pipeline y aprovechar su `fit_transform` con cross-fitting.

### Overfitting
- Se utiliza validación cruzada estratificada.
- Se compara score de entrenamiento contra validación.
- Se reserva un test final.

### Desbalance
- Se reportan precision, recall y F1 además de accuracy.
- Se usa `class_weight="balanced"` en los modelos de referencia.
- El umbral se puede ajustar según los costos de negocio.

### Dependencia temporal
Este dataset no está planteado como una serie temporal longitudinal. Si el problema real tuviera observaciones ordenadas por fecha y el objetivo fuera predecir churn futuro, debería utilizarse una estrategia temporal en lugar de una validación aleatoria.

# 24. Conclusiones — plantilla de redacción

Al ejecutar el notebook, reemplazar los campos entre corchetes por los resultados reales.

**Contexto:** se abordó un problema de clasificación binaria orientado a detectar clientes con riesgo de churn.

**Calidad de datos:** el dataset contiene [N] registros y [M] variables. Se detectaron [X] valores faltantes, principalmente en [variable]. `TotalCharges` requirió conversión a numérico y los valores faltantes fueron tratados dentro del pipeline.

**EDA:** la tasa de churn observada fue de [X]%. Las variables que mostraron diferencias descriptivas más marcadas entre clases fueron [variables]. Estas asociaciones no deben interpretarse como causalidad.

**Preprocesamiento:** las variables numéricas fueron imputadas con mediana y estandarizadas; las categóricas fueron imputadas y codificadas mediante One-Hot Encoding. El identificador `customerID` fue excluido por no representar una característica generalizable.

**Modelado:** la regresión logística se utilizó como baseline y Random Forest como modelo no lineal. La búsqueda de hiperparámetros se realizó con GridSearchCV y StratifiedKFold optimizando ROC AUC.

**Evaluación:** el modelo final obtuvo ROC AUC = [X], accuracy = [X], precision = [X], recall = [X] y F1 = [X] sobre el conjunto de test.

**Negocio:** el modelo puede utilizarse como sistema de priorización de clientes. Antes de desplegarlo sería necesario definir el costo de contacto, tasa de recuperación y valor económico de un cliente retenido. El umbral de clasificación debería seleccionarse según esos costos, no únicamente según una métrica estadística.

**Limitaciones:** el dataset es una muestra de referencia y no representa necesariamente una operación real actual. Las asociaciones observadas no prueban causalidad y el rendimiento obtenido en este dataset no garantiza el mismo rendimiento en producción.

# 25. Checklist de entrega

- [ ] Contexto y problema de negocio.
- [ ] Fuente y descripción del dataset.
- [ ] Dimensiones y tipos.
- [ ] Calidad de datos y missingness.
- [ ] Balance de `Churn`.
- [ ] EDA numérico y categórico.
- [ ] Feature engineering justificado.
- [ ] Split train/test estratificado.
- [ ] Pipeline de preprocesamiento.
- [ ] Baseline.
- [ ] Random Forest.
- [ ] GridSearchCV + StratifiedKFold.
- [ ] ROC AUC.
- [ ] Accuracy.
- [ ] Precision / Recall / F1.
- [ ] Matriz de confusión.
- [ ] Curva ROC.
- [ ] Interpretabilidad.
- [ ] Impacto económico parametrizado.
- [ ] Limitaciones y riesgos.
- [ ] Conclusión.